# Intro

## Global parameters

In [1]:
MAX_TIME_HOURS=1
ALPHA=0.01
_PVAL_FLOOR=10**-6

## Modules

### Standard

In [2]:
import os, pickle, platform, sys
import numpy as np

In [3]:
from collections import defaultdict

In [4]:
import dcms
from dcms.models import DCMModel, DECMModel, qDECMModel, DWCMModel

In [5]:
import matplotlib.pyplot as plt
plt.rcParams['axes.linewidth'] = 2
plt.rcParams['xtick.major.size'] = 10
plt.rcParams['xtick.major.width'] = 2
plt.rcParams['ytick.major.size'] = 10
plt.rcParams['ytick.major.width'] = 2

plt.rcParams['xtick.labelsize'] = 14
plt.rcParams['ytick.labelsize'] = 14

plt.rcParams['xtick.minor.size'] = 5
plt.rcParams['xtick.minor.width'] = 1
plt.rcParams['ytick.minor.size'] = 5
plt.rcParams['ytick.minor.width'] = 1
from matplotlib.lines import Line2D
import matplotlib.colors as mcolors
from matplotlib.ticker import ScalarFormatter

In [6]:
from scipy.stats import spearmanr

In [7]:
from tqdm.notebook import tqdm, trange

In [8]:
import datetime as dt

In [9]:
from bowtie import edges2bowtie

### Home made

In [10]:
if platform.system() == 'Darwin':
    print('Air!')
    HOME = '/Users/fabio/Documents/Lavoro/PythonFiles/bowtie2_py310/bowtie2/'
elif platform.system() == 'Linux':
    print('Stella!')
    HOME = '/home/sarawalk/bowtie2_py39/bowtie2/'
else:
    raise RuntimeError(f"Unsupported OS: {platform.system()}")

sys.path.insert(0, HOME)

Air!


In [11]:
from auxiliary_functions import el2ks

In [12]:
from sam_bowtie import block_and_fluxes as bnf

In [13]:
from plot_bowtie import plot_bowtie_blocks, plot_bowtie_fluxes, _add_colorbar
from plot_bowtie import _fdr as fdr

## Load data

In [14]:
DATA_FOLDER=HOME+'dati_elezioni/'
TEST_FOLDER=HOME+'tests/'
PVALUE_FOLDER=HOME+'pvalues/'
GUARINO_FOLDER=HOME+'guarino_files/'
BIPARTITE_FOLDER_1=HOME+'BiDCM/'
BIPARTITE_FOLDER_2=HOME+'BiDCM2/'
PLOT_FOLDER=HOME+'plots/'

## Looking into the abyss

In [15]:
guarino_files=[file for file in os.listdir(GUARINO_FOLDER) if not file.startswith('.')]
guarino_files.sort()
guarino_files

['all_dico_labels.txt',
 'crisi_dico_0_bowtie_sizes.csv',
 'crisi_dico_1_bowtie_sizes.csv',
 'crisi_dico_2_bowtie_sizes.csv',
 'crisi_dico_3_bowtie_sizes.csv',
 'crisi_dico_4_bowtie_sizes.csv',
 'crisi_dico_labels.pickle',
 'ita_elections_dico_0_bowtie_sizes.csv',
 'ita_elections_dico_1_bowtie_sizes.csv',
 'ita_elections_dico_2_bowtie_sizes.csv',
 'ita_elections_dico_3_bowtie_sizes.csv',
 'ita_elections_dico_4_bowtie_sizes.csv',
 'ita_elections_dico_5_bowtie_sizes.csv',
 'ita_elections_dico_6_bowtie_sizes.csv',
 'ita_elections_dico_labels.pickle',
 'quirinale_dico_0_bowtie_sizes.csv',
 'quirinale_dico_1_bowtie_sizes.csv',
 'quirinale_dico_2_bowtie_sizes.csv',
 'quirinale_dico_3_bowtie_sizes.csv',
 'quirinale_dico_4_bowtie_sizes.csv',
 'quirinale_dico_5_bowtie_sizes.csv',
 'quirinale_dico_6_bowtie_sizes.csv',
 'quirinale_dico_labels.pickle']

In [16]:
bipartite_files_1=[file for file in os.listdir(BIPARTITE_FOLDER_1) if not file.startswith('.')]
bipartite_files_1.sort()
bipartite_files_1
bipartite_files_2=[file for file in os.listdir(BIPARTITE_FOLDER_2) if not file.startswith('.')]
bipartite_files_2.sort()
bipartite_files_2

['crisi_dico_0_bowtie_flows.csv',
 'crisi_dico_0_bowtie_flows_recomputed_sectors.csv',
 'crisi_dico_0_bowtie_sizes.csv',
 'crisi_dico_1_bowtie_flows.csv',
 'crisi_dico_1_bowtie_flows_recomputed_sectors.csv',
 'crisi_dico_1_bowtie_sizes.csv',
 'crisi_dico_2_bowtie_flows.csv',
 'crisi_dico_2_bowtie_flows_recomputed_sectors.csv',
 'crisi_dico_2_bowtie_sizes.csv',
 'crisi_dico_3_bowtie_flows.csv',
 'crisi_dico_3_bowtie_flows_recomputed_sectors.csv',
 'crisi_dico_3_bowtie_sizes.csv',
 'crisi_dico_4_bowtie_flows.csv',
 'crisi_dico_4_bowtie_flows_recomputed_sectors.csv',
 'crisi_dico_4_bowtie_sizes.csv',
 'ita_elections_dico_0_bowtie_flows.csv',
 'ita_elections_dico_0_bowtie_flows_recomputed_sectors.csv',
 'ita_elections_dico_0_bowtie_sizes.csv',
 'ita_elections_dico_1_bowtie_flows.csv',
 'ita_elections_dico_1_bowtie_flows_recomputed_sectors.csv',
 'ita_elections_dico_1_bowtie_sizes.csv',
 'ita_elections_dico_2_bowtie_flows.csv',
 'ita_elections_dico_2_bowtie_flows_recomputed_sectors.csv',
 '

### Name of the various dicos

In [17]:
guarino_files[0]

'all_dico_labels.txt'

In [18]:
with open(GUARINO_FOLDER+guarino_files[0], 'r') as f:
    cacca=f.readlines()


In [19]:
def parse_dico_names(filepath):
    with open(filepath, 'r') as f:
        lines = [line.strip().split() for line in f]
        # such a command creates a list
        # in which each element is a list of tokens of a line in the file
        # remarkably, there is an empty element
        # before each dataset name
        

    result = {}
    current_key = None

    for tokens in lines:
        if not tokens:
            current_key = None
        elif current_key is None:
            # prima riga non vuota del gruppo: nome del dataset
            current_key = tokens[0]
            result[current_key] = {}
        else:
            # riga tipo ['5:', 'journalists', '&', 'Media']
            idx = int(tokens[0].rstrip(':'))
            label = ' '.join(tokens[1:])
            result[current_key][idx] = label

    return result

In [20]:
cacca=parse_dico_names(GUARINO_FOLDER+guarino_files[0])

In [21]:
with open(GUARINO_FOLDER+guarino_files[6], 'rb') as f:
    cacca = pickle.load(f)

In [22]:
cacca

{1: 'Lega & FDI & FI',
 2: 'M5S & journalists',
 0: 'journalists & IV & Media & Azione & +Europa',
 4: 'Media & journalists',
 3: 'PD'}

The pickle is what I need.

## Functions

### guarino2dict_blocks

In [ ]:
def guarino2dict_blocks(dataset, dico, folder):
    # get the file
    file_name=f'{dataset}_dico_{dico}_bowtie_sizes.csv'
    # load the data
    BIPARTITE_FOLDER=BIPARTITE_FOLDER_1 if folder==1 else BIPARTITE_FOLDER_2
    cacca=np.genfromtxt(BIPARTITE_FOLDER+file_name, delimiter=',', dtype=int, skip_header=1)
    header = np.genfromtxt(BIPARTITE_FOLDER+file_name, delimiter=',', dtype=str, max_rows=1)
    header=[str(h) for h in header]
    # load the "monopartite" dict
    block_dict_0, flux_dict_0 = ppmf(dataset, dico)
    
    block_dict_bipartite={}

    guarino_translator={'LSCC':'SCC', 'IN-TENDRILS':'INTENDRILS', 'OUT-TENDRILS':'OUTTENDRILS'}

    for i, name in enumerate(header):
        key=guarino_translator.get(name, name)
        block_dict_bipartite[key] = {}
        block_dict_bipartite[key]['obs']=block_dict_0[key]['obs']
        block_dict_bipartite[key]['sample']=cacca[:,i]
        _ge=np.sum(cacca[:,i]>=block_dict_bipartite[key]['obs'])
        _le=np.sum(cacca[:,i]<=block_dict_bipartite[key]['obs'])
        block_dict_bipartite[key]['p_value']=2*min(_ge, _le)/len(block_dict_bipartite[key]['sample'])
        _median=np.median(block_dict_bipartite[key]['sample'])
        if _median>block_dict_bipartite[key]['obs']:
            block_dict_bipartite[key]['tail']='left'
        else:
            block_dict_bipartite[key]['tail']='right'
    return block_dict_bipartite, block_dict_0, flux_dict_0



### guarino2dict_fluxes

In [24]:
def guarino2dict_fluxes(dataset, dico):
    # get the file
    file_name=f'{dataset}_dico_{dico}_bowtie_flows.csv'
    # load the data
    cacca=np.genfromtxt(BIPARTITE_FOLDER+file_name, delimiter=',', skip_header=1)
    cacca=cacca.astype(int)
    header = np.genfromtxt(BIPARTITE_FOLDER+file_name, delimiter=',', dtype=str, max_rows=1)

    _, flux_dict_0 = ppmf(dataset, dico)

    flux_dict_bipartite=defaultdict(dict)
    guarino_translator={'LSCC':'SCC', 'IN-TENDRILS':'INTENDRILS', 'OUT-TENDRILS':'OUTTENDRILS'}

    

    for i, name in enumerate(header):
        source, target = name.split('->')
        # translate the source and target to match the keys in flux_dict_0
        source = guarino_translator.get(source, source)
        target = guarino_translator.get(target, target)
        key = '->'.join([source, target])

        flux_dict_bipartite[key]['obs']=flux_dict_0[key]['obs']
        flux_dict_bipartite[key]['sample']=cacca[:,i]
        _ge=np.sum(cacca[:,i]>=flux_dict_bipartite[key]['obs'])
        _le=np.sum(cacca[:,i]<=flux_dict_bipartite[key]['obs'])
        flux_dict_bipartite[key]['p_value']=2*min(_ge, _le)/len(flux_dict_bipartite[key]['sample'])
        _median=np.median(flux_dict_bipartite[key]['sample'])
        if _median>flux_dict_bipartite[key]['obs']:
            flux_dict_bipartite[key]['tail']='left'
        else:
            flux_dict_bipartite[key]['tail']='right'
        
    return flux_dict_bipartite, flux_dict_0
    

### right_tailer

In [25]:
def right_tailer(block_dict):
    right_tailed_dict={}
    for key, item in block_dict.items():
        right_tailed_dict[key] = item
        if item['tail'] == 'right':
            right_tailed_dict[key]['p_value'] /=2
        else:
            right_tailed_dict[key]['p_value']=1.
    return right_tailed_dict


### ppmf: pre-processing my files

In [26]:
def ppmf(dataset, dico):
    with open(PVALUE_FOLDER+f'{dataset}_dico{dico}_pvalues_blocks_1.pkl', 'rb') as f:
        block_dict_1 = pickle.load(f)

    with open(PVALUE_FOLDER+f'{dataset}_dico{dico}_pvalues_fluxes_1.pkl', 'rb') as f:
        flux_dict_1 = pickle.load(f)

    for key, value in block_dict_1.items():
        block_dict_1[key]['mean_sim']=np.mean(value['count_sample'])
        block_dict_1[key]['std_sim']=np.std(value['count_sample'])    
        if np.median(value['count_sample']) > block_dict_1[key]['obs']:
            block_dict_1[key]['tail']='left'
        else:
            block_dict_1[key]['tail']='right'
        
    for key, value in flux_dict_1.items():
        flux_dict_1[key]['mean_sim']=np.mean(value['count_sample'])
        flux_dict_1[key]['std_sim']=np.std(value['count_sample'])    
        if np.median(value['count_sample']) > flux_dict_1[key]['obs']:
            flux_dict_1[key]['tail']='left'
        else:
            flux_dict_1[key]['tail']='right'    

    return block_dict_1, flux_dict_1


#### guarino2dict_blocks_DCM

In [ ]:
def guarino2dict_blocks_DCM(dataset, dico):
    # get the file
    file_name=f'{dataset}_dico_{dico}_bowtie_sizes.csv'
    # load the data
    cacca=np.genfromtxt(GUARINO_FOLDER+file_name, delimiter=',', dtype=int, skip_header=1)
    header = np.genfromtxt(GUARINO_FOLDER+file_name, delimiter=',', dtype=str, max_rows=1)
    header=[str(h) for h in header]
    # load the "monopartite" dict
    with open(PVALUE_FOLDER+f'{dataset}_dico{dico}_pvalues_blocks_0.pkl', 'rb') as f:
        block_dict_0 = pickle.load(f)

    with open(PVALUE_FOLDER+f'{dataset}_dico{dico}_pvalues_fluxes_0.pkl', 'rb') as f:
        flux_dict_0 = pickle.load(f)

    block_dict_bipartite={}

    for i, name in enumerate(header):
        if name=='LSCC':
            key='SCC'
        else:
            key=name.replace('-','')
        block_dict_bipartite[key] = {}
        block_dict_bipartite[key]['obs']=block_dict_0[key]['obs']
        block_dict_bipartite[key]['sample']=cacca[:,i]
        _ge=np.sum(cacca[:,i]>=block_dict_bipartite[key]['obs'])
        _le=np.sum(cacca[:,i]<=block_dict_bipartite[key]['obs'])
        block_dict_bipartite[key]['p_value']=2*min(_ge, _le)/len(block_dict_bipartite[key]['sample'])
        _median=np.median(block_dict_bipartite[key]['sample'])
        if _median>block_dict_bipartite[key]['obs']:
            block_dict_bipartite[key]['tail']='left'
        else:
            block_dict_bipartite[key]['tail']='right'
    return block_dict_bipartite, block_dict_0, flux_dict_0



# Bipartite 1, Bipartite 2

### Prelude

In [28]:
labels_file=[file for file in os.listdir(GUARINO_FOLDER) if file.endswith('.pickle') and file.startswith('ita')][0]

In [29]:
labels=pickle.load(open(GUARINO_FOLDER+labels_file, 'rb'))

In [30]:
labels

{1: 'PD & Media & +Europa & journalists',
 2: 'M5S & Media',
 3: 'journalists & IV & Azione',
 0: 'Lega & FDI & FI',
 4: 'journalists & Media (1)',
 5: 'Media',
 6: 'journalists & Media (2)'}

### Right-wing

In [31]:
dataset='ita_elections'
dico=0

#### Bowtie

In [ ]:
blocks=guarino2dict_blocks(dataset, dico)
blocks_dcm=guarino2dict_blocks_DCM(dataset, dico)
fluxes=guarino2dict_fluxes(dataset, dico)

In [ ]:
users=0
for key in blocks[0].keys():
    users+=blocks[0][key]['obs']

In [ ]:
rts=0
for key in blocks[2].keys():
    rts+=blocks[2][key]['obs']

In [ ]:
users, rts

(50288, 688718)